In [ ]:
# import json
from abc import ABC, abstractmethod

# =====================================================================
# 1. ABSTRACTION (Soyutlama) & BASE CLASS (Temel Sınıf)
# =====================================================================
class TemelSistemElemani(ABC):
    """
    Sistemdeki kimlik sahibi (Ürün, Kullanıcı vb.) tüm sınıflar için 
    ortak bir soyut temel sınıf oluşturarak Abstraction sağladık.
    """
    def __init__(self, id_no):
        self._id_no = id_no  # Encapsulation için protected üye

    @abstractmethod
    def bilgi_goster(self):
        """Polymorphism sağlamak için soyut metot tanımı."""
        pass


# =====================================================================
# 2. ENCAPSULATION & CLASS-OBJECT YAPILARI
# =====================================================================
class Urun(TemelSistemElemani):
    def __init__(self, urun_id, ad, fiyat, stok):
        super().__init__(urun_id)
        self.__ad = ad            # Private attribute (Encapsulation)
        self.__fiyat = fiyat      
        self.__stok = stok       

    # Getter ve Setter Metotları (Kapsülleme Kontrolleri)
    def get_id(self): return self._id_no
    def get_ad(self): return self.__ad
    def get_fiyat(self): return self.__fiyat
    def get_stok(self): return self.__stok
    
    def set_stok(self, yeni_stok):
        if yeni_stok >= 0:
            self.__stok = yeni_stok
        else:
            raise ValueError("Stok miktarı negatif olamaz!")

    def bilgi_goster(self):
        """Polymorphism Uygulaması"""
        return f"[ID: {self.get_id()}] {self.__ad} - Fiyat: {self.__fiyat} TL | Stok: {self.__stok} adet"


# =====================================================================
# 3. INHERITANCE (Kalıtım) & POLYMORPHISM (Çok Biçimlilik)
# =====================================================================
class Kullanici(TemelSistemElemani):
    """Temel Kullanıcı Sınıfı"""
    def __init__(self, kullanici_id, isim, eposta):
        super().__init__(kullanici_id)
        self.isim = isim
        self.eposta = eposta
        self.sepet = Sepet() # Her kullanıcının bir sepet nesnesi vardır (Composition)

    def bilgi_goster(self):
        """Polymorphism Uygulaması"""
        return f"Standart Üye -> {self.isim} ({self.eposta})"

    def kargo_ucreti_hesapla(self, alt_toplam):
        """Standart kullanıcılar için kargo ücreti politikası"""
        if alt_toplam >= 500:
            return 0  # 500 TL üzeri kargo bedava
        return 50.0   # Altı için 50 TL kargo ücreti


class PremiumKullanici(Kullanici):
    """Kullanıcı sınıfından türetilmiş Kalıtım (Inheritance) örneği"""
    def __init__(self, kullanici_id, isim, eposta, premium_kod):
        super().__init__(kullanici_id, isim, eposta)
        self.premium_kod = premium_kod

    def bilgi_goster(self):
        """Metot Ezme (Method Overriding) ile Polymorphism"""
        return f"🌟 Premium Üye -> {self.isim} ({self.eposta}) | Kod: {self.premium_kod}"

    def kargo_ucreti_hesapla(self, alt_toplam):
        """Premium üyelere özel kargo her zaman bedava (Polymorphism)"""
        return 0.0


# =====================================================================
# 4. SEPET VE SİPARİŞ MANTIĞI (İş Kuralları Sınıfları)
# =====================================================================
class Sepet:
    def __init__(self):
        # {urun_id: miktar} şeklinde sözlük (Dictionary) kullanımı
        self.urunler = {}

    def urun_ekle(self, urun_id, miktar=1):
        if urun_id in self.urunler:
            self.urunler[urun_id] += miktar
        else:
            self.urunler[urun_id] = miktar

    def urun_cikar(self, urun_id):
        if urun_id in self.urunler:
            del self.urunler[urun_id]
            return True
        return False

    def temizle(self):
        self.urunler.clear()


class Siparis:
    def __init__(self, siparis_id, kullanici_isim, urunler_ozet, toplam_tutar):
        self.siparis_id = siparis_id
        self.kullanici_isim = kullanici_isim
        self.urunler_ozet = urunler_ozet  # Sipariş anındaki ürün listesi metni
        self.toplam_tutar = toplam_tutar

    def siparis_bilgisi(self):
        return f"Sipariş No: {self.siparis_id} | Alıcı: {self.kullanici_isim} | Ürünler: {self.urunler_ozet} | Toplam: {self.toplam_tutar} TL"


# =====================================================================
# 5. ANA SİSTEM YÖNETİMİ (Dosya İşlemleri & Menü Motoru)
# =====================================================================
class OnlineAlisverisSistemi:
    def __init__(self):
        self.urunler_listesi = {}      # {urun_id: Urun Nesnesi}
        self.kullanicilar_listesi = {} # {kullanici_id: Kullanici/Premium Nesnesi}
        self.siparisler_listesi = []   # Siparis nesneleri listesi (List)
        self.siparis_sayaci = 1001

    # VERİ KAYDETME (JSON Dosya İşlemleri)
    def verileri_kaydet(self, dosya_adi="data.json"):
        try:
            data = {
                "urunler": [],
                "kullanicilar": [],
                "siparisler": []
            }
            # Ürünleri sözlüğe çevir
            for u in self.urunler_listesi.values():
                data["urunler"].append({"id": u.get_id(), "ad": u.get_ad(), "fiyat": u.get_fiyat(), "stok": u.get_stok()})
            
            # Kullanıcıları sözlüğe çevir
            for k in self.kullanicilar_listesi.values():
                u_tipi = "Premium" if isinstance(k, PremiumKullanici) else "Standart"
                p_kod = k.premium_kod if u_tipi == "Premium" else ""
                data["kullanicilar"].append({
                    "id": k._id_no, "isim": k.isim, "eposta": k.eposta, "tip": u_tipi, "premium_kod": p_kod
                })
            
            # Siparişleri sözlüğe çevir
            for s in self.siparisler_listesi:
                data["siparisler"].append({
                    "id": s.siparis_id, "kullanici": s.kullanici_isim, "ozet": s.urunler_ozet, "tutar": s.toplam_tutar
                })

            with open(dosya_adi, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
            print(f"✅ Veriler '{dosya_adi}' dosyasına başarıyla kaydedildi.")
        except Exception as e:
            print(f"❌ Kaydedilirken hata oluştu: {e}")

    # VERİ YÜKLEME (JSON Dosya İşlemleri)
    def verileri_yukle(self, dosya_adi="data.json"):
        try:
            with open(dosya_adi, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            self.urunler_listesi.clear()
            self.kullanicilar_listesi.clear()
            self.siparisler_listesi.clear()

            for u in data.get("urunler", []):
                self.urunler_listesi[u["id"]] = Urun(u["id"], u["ad"], u["fiyat"], u["stok"])
            
            for k in data.get("kullanicilar", []):
                if k["tip"] == "Premium":
                    self.kullanicilar_listesi[k["id"]] = PremiumKullanici(k["id"], k["isim"], k["eposta"], k["premium_kod"])
                else:
                    self.kullanicilar_listesi[k["id"]] = Kullanici(k["id"], k["isim"], k["eposta"])
            
            for s in data.get("siparisler", []):
                self.siparisler_listesi.append(Siparis(s["id"], s["kullanici"], s["ozet"], s["tutar"]))
                
            print(f"📂 Veriler '{dosya_adi}' dosyasından başarıyla yüklendi.")
        except FileNotFoundError:
            print("ℹ️ Kayıtlı veri dosyası bulunamadı. Temiz sistem başlatılıyor.")
        except Exception as e:
            print(f"❌ Yüklenirken hata oluştu: {e}")


# =====================================================================
# INTERACTIVE TERMINAL MENU RUNNER
# =====================================================================
def ana_akis():
    sistem = OnlineAlisverisSistemi()
    # Varsayılan başlangıç verileri ekleyelim (Kolay test için)
    sistem.urunler_listesi["U1"] = Urun("U1", "Kablosuz Kulaklık", 1200.0, 10)
    sistem.urunler_listesi["U2"] = Urun("U2", "Akıllı Saat", 2500.0, 5)
    sistem.urunler_listesi["U3"] = Urun("U3", "Kahve Kupası", 150.0, 50)
    
    sistem.kullanicilar_listesi["K1"] = Kullanici("K1", "Ahmet Yılmaz", "ahmet@mail.com")
    sistem.kullanicilar_listesi["K2"] = PremiumKullanici("K2", "Elif Demir", "elif@mail.com", "PREM2026")

    while True:
        print("\n" + "="*35)
        print("      ONLINE ALIŞVERİŞ SİSTEMİ     ")
        print("="*35)
        print("1- Ürün Ekle")
        print("2- Ürünleri Listele")
        print("3- Kullanıcı Ekle")
        print("4- Sepete Ürün Ekle")
        print("5- Sepetten Ürün Çıkar")
        print("6- Sepeti Görüntüle")
        print("7- Sipariş Oluştur (Satın Al)")
        print("8- Siparişleri Listele")
        print("9- Toplam Sepet Tutarını Hesapla")
        print("10- Verileri Dosyaya Kaydet")
        print("11- Verileri Dosyadan Yükle")
        print("0- Çıkış")
        print("="*35)
        
        secim = input("Lütfen yapmak istediğiniz işlemi seçiniz (0-11): ").strip()
        
        # TRY-EXCEPT İLE HATA YÖNETİMİ
        try:
            if secim == "1":
                print("\n--- YENİ ÜRÜN EKLE ---")
                uid = input("Ürün ID (örn: U4): ").strip()
                if uid in sistem.urunler_listesi:
                    print("❌ Bu ID ile zaten bir ürün mevcut!")
                    continue
                ad = input("Ürün Adı: ").strip()
                fiyat = float(input("Fiyatı (TL): "))
                stok = int(input("Stok Adedi: "))
                sistem.urunler_listesi[uid] = Urun(uid, ad, fiyat, stok)
                print(f"✅ {ad} başarıyla sisteme eklendi.")

            elif secim == "2":
                print("\n--- SİSTEMDEKİ ÜRÜNLER ---")
                if not sistem.urunler_listesi:
                    print("Sistemde kayıtlı ürün bulunmamaktadır.")
                for urun in sistem.urunler_listesi.values():
                    print(urun.bilgi_goster())

            elif secim == "3":
                print("\n--- YENİ KULLANICI EKLE ---")
                kid = input("Kullanıcı ID (örn: K3): ").strip()
                if kid in sistem.kullanicilar_listesi:
                    print("❌ Bu ID ile zaten bir kullanıcı mevcut!")
                    continue
                isim = input("Ad Soyad: ").strip()
                eposta = input("E-Posta: ").strip()
                tip = input("Üyelik Tipi (1: Standart, 2: Premium): ").strip()
                
                if tip == "2":
                    kod = input("Premium Kampanya Kodu: ").strip()
                    sistem.kullanicilar_listesi[kid] = PremiumKullanici(kid, isim, eposta, kod)
                else:
                    sistem.kullanicilar_listesi[kid] = Kullanici(kid, isim, eposta)
                print(f"✅ {isim} kullanıcısı sisteme tanımlandı.")

            elif secim == "4":
                print("\n--- SEPETE ÜRÜN EKLE ---")
                kid = input("Kullanıcı ID girin: ").strip()
                if kid not in sistem.kullanicilar_listesi:
                    print("❌ Kullanıcı bulunamadı!")
                    continue
                uid = input("Eklenecek Ürün ID girin: ").strip()
                if uid not in sistem.urunler_listesi:
                    print("❌ Ürün bulunamadı!")
                    continue
                adet = int(input("Kaç adet eklenecek?: "))
                
                # ÜRÜN STOK KONTROLÜ
                if sistem.urunler_listesi[uid].get_stok() < adet:
                    print(f"❌ Yetersiz stok! Mevcut stok: {sistem.urunler_listesi[uid].get_stok()}")
                else:
                    sistem.kullanicilar_listesi[kid].sepet.urun_ekle(uid, adet)
                    print("✅ Ürün sepete eklendi.")

            elif secim == "5":
                print("\n--- SEPETTEN ÜRÜN ÇIKAR ---")
                kid = input("Kullanıcı ID girin: ").strip()
                if kid in sistem.kullanicilar_listesi:
                    uid = input("Çıkarılacak Ürün ID girin: ").strip()
                    if sistem.kullanicilar_listesi[kid].sepet.urun_cikar(uid):
                        print("✅ Ürün sepetten tamamen kaldırıldı.")
                    else:
                        print("❌ Ürün zaten sepette yok.")
                else:
                    print("❌ Kullanıcı bulunamadı.")

            elif secim == "6":
                print("\n--- SEPET GÖRÜNTÜLE ---")
                kid = input("Kullanıcı ID girin: ").strip()
                if kid not in sistem.kullanicilar_listesi:
                    print("❌ Kullanıcı bulunamadı!")
                    continue
                kullanici = sistem.kullanicilar_listesi[kid]
                print(f"🛒 {kullanici.isim} isimli kullanıcının Sepeti:")
                if not kullanici.sepet.urunler:
                    print("Sepetiniz tamamen boş.")
                for uid, miktar in kullanici.sepet.urunler.items():
                    u = sistem.urunler_listesi[uid]
                    print(f"   - {u.get_ad()} | Adet: {miktar} | Birim Fiyat: {u.get_fiyat()} TL")

            elif secim == "7":
                print("\n--- SİPARİŞ OLUŞTUR (SATIN AL) ---")
                kid = input("Siparişi tamamlayacak Kullanıcı ID: ").strip()
                if kid not in sistem.kullanicilar_listesi:
                    print("❌ Kullanıcı bulunamadı!")
                    continue
                
                kullanici = sistem.kullanicilar_listesi[kid]
                if not kullanici.sepet.urunler:
                    print("❌ Sepetiniz boş olduğundan sipariş oluşturulamaz.")
                    continue
                
                # Stok kontrolü ve sipariş hesabı ortak yürütülür
                stok_sorunu = False
                alt_toplam = 0.0
                ozet_listesi = []
                
                for uid, miktar in kullanici.sepet.urunler.items():
                    u = sistem.urunler_listesi[uid]
                    if u.get_stok() < miktar:
                        print(f"❌ Sipariş başarısız! {u.get_ad()} için yetersiz stok var (Mevcut: {u.get_stok()}).")
                        stok_sorunu = True
                        break
                    alt_toplam += u.get_fiyat() * miktar
                    ozet_listesi.append(f"{u.get_ad()}(x{miktar})")
                
                if stok_sorunu:
                    continue
                
                # Kargo ve Toplam Tutar İndirim Mantığı (Polymorphism kullanımı)
                kargo = kullanici.kargo_ucreti_hesapla(alt_toplam)
                toplam_tutar = alt_toplam + kargo
                
                # Stokları düşelim
                for uid, miktar in kullanici.sepet.urunler.items():
                    u = sistem.urunler_listesi[uid]
                    u.set_stok(u.get_stok() - miktar)
                
                # Siparişi kaydet
                ozet_metni = ", ".join(ozet_listesi)
                yeni_siparis = Siparis(sistem.siparis_sayaci, kullanici.isim, ozet_metni, toplam_tutar)
                sistem.siparisler_listesi.append(yeni_siparis)
                
                print(f"🎉 Siparişiniz Alındı! Kargo: {kargo} TL | Toplam Ödenen: {toplam_tutar} TL")
                sistem.siparis_sayaci += 1
                kullanici.sepet.temizle() # Sipariş bitince sepet boşalır

            elif secim == "8":
                print("\n--- TÜM SİPARİŞLER (RAPOR) ---")
                if not sistem.siparisler_listesi:
                    print("Sistemde henüz gerçekleşmiş sipariş bulunmuyor.")
                for s in sistem.siparisler_listesi:
                    print(s.siparis_id_no if hasattr(s, 'siparis_id_no') else s.siparis_bilgisi())

            elif secim == "9":
                print("\n--- ANLIK SEPET TUTARI ---")
                kid = input("Kullanıcı ID girin: ").strip()
                if kid in sistem.kullanicilar_listesi:
                    kullanici = sistem.kullanicilar_listesi[kid]
                    alt_toplam = sum(sistem.urunler_listesi[uid].get_fiyat() * miktar for uid, miktar in kullanici.sepet.urunler.items())
                    kargo = kullanici.kargo_ucreti_hesapla(alt_toplam)
                    print(f"Ürünler Toplamı: {alt_toplam} TL")
                    print(f"Kargo Ücreti: {kargo} TL")
                    print(f"Ödenecek Net Tutar: {alt_toplam + kargo} TL")
                else:
                    print("❌ Kullanıcı bulunamadı.")

            elif secim == "10":
                sistem.verileri_kaydet()

            elif secim == "11":
                sistem.verileri_yukle()

            elif secim == "0":
                print("👋 Programdan çıkılıyor. İyi günler!")
                break
            else:
                print("❌ Geçersiz seçim! Lütfen 0-11 arası bir değer girin.")
                
        except ValueError as ve:
            print(f"⚠️ Girdi Hatası: Sayısal değerleri doğru girdiğinizden emin olun! ({ve})")
        except Exception as e:
            print(f"⚠️ Beklenmedik bir hata oluştu: {e}")

if __name__ == "__main__":
    ana_akis()


      ONLINE ALIŞVERİŞ SİSTEMİ     
1- Ürün Ekle
2- Ürünleri Listele
3- Kullanıcı Ekle
4- Sepete Ürün Ekle
5- Sepetten Ürün Çıkar
6- Sepeti Görüntüle
7- Sipariş Oluştur (Satın Al)
8- Siparişleri Listele
9- Toplam Sepet Tutarını Hesapla
10- Verileri Dosyaya Kaydet
11- Verileri Dosyadan Yükle
0- Çıkış
